# MuseTalk v1.5 — Arabic lip-sync repair (resumable, budget-capped)

This notebook works on the **verified 1h42m dubbed file** and reuses the ten existing Arabic voice masters. It does not call Gemini, regenerate TTS, or run ASR again.

**Important:** the initial alignment is imperfect. The strict planner retains only high-confidence, non-overrunning, non-overlapping utterances. Current report: 152 rendered lines; 49 safe lines; 113 clips after splitting; about 12.4 minutes of dialogue coverage (roughly 12% of the episode). This is a partial safe pass, not a claim that the whole episode is lip-synced.

Budget controls: T4-or-better GPU required, maximum 8 seconds per clip, 6 clips per session, 25 minutes of GPU work per session. First run is pilot-only. Review the pilot before enabling batches. Completed clips and model weights are cached on Drive so a disconnected session can resume.

MuseTalk source code is MIT-licensed; its bundled dependencies and pretrained weights have separate licenses. Verify each model card and dependency license before commercial distribution.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, hashlib, shutil, subprocess, zipfile, requests, sys, math
from pathlib import Path

REPO = 'mhommdkuyr/AI-Creative-Studio'
BRANCH = 'fix/srt-cue-dedup-and-sync-audit'
RELEASE_TAG = 'arabic-dub-assets-2026-10-31'
BASE_VIDEO_ID = '15MkBM0m9YduRz4pim2Stp_9ruW5XbqKw'
OUTPUT_FOLDER_ID = '1SE3JYbB4c-pC1W-ymRK6S5CsFt-XidzA'
CACHE = Path('/content/drive/MyDrive/MuseTalkArabicCache')
CACHE.mkdir(parents=True, exist_ok=True)
os.environ['CACHE'] = str(CACHE)
WORK = Path('/content/musetalk_work'); WORK.mkdir(exist_ok=True)

# Hard limits — change only after reviewing the pilot and available Colab quota.
PILOT_ONLY = True
MAX_CLIPS_PER_SESSION = 6
MAX_SESSION_GPU_MINUTES = 25
MAX_CLIP_SECONDS = 8.0
MIN_CONFIDENCE = 0.80
MIN_WORD_COVERAGE = 0.70
subprocess.run([sys.executable,'-m','pip','install','-q','pyyaml'],check=True)


In [ ]:
# Stop immediately if this runtime has no suitable GPU. Never fall back to CPU inference.
subprocess.run(['nvidia-smi'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU unavailable. Stop here; do not run MuseTalk on CPU.')
gpu = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print('GPU:', gpu, 'VRAM_GB:', round(mem_gb, 1))
if mem_gb < 10:
    raise RuntimeError('Need at least 10 GB VRAM for the capped MuseTalk v1.5 pilot.')


In [ ]:
# Download the verified repair-only release asset (10 voice masters + exact alignment/report).
api = f'https://api.github.com/repos/{REPO}/releases/tags/{RELEASE_TAG}'
r = requests.get(api, timeout=60); r.raise_for_status(); release = r.json()
asset = next(a for a in release['assets'] if a['name'] == 'arabic_dub_lipsync_repair_inputs.zip')
bundle = WORK / 'repair_inputs.zip'
if not bundle.exists() or bundle.stat().st_size != asset['size']:
    with requests.get(asset['browser_download_url'], stream=True, timeout=120) as resp:
        resp.raise_for_status()
        with open(bundle, 'wb') as f:
            for chunk in resp.iter_content(8*1024*1024):
                if chunk: f.write(chunk)
print('Bundle bytes:', bundle.stat().st_size)
with zipfile.ZipFile(bundle) as z: z.extractall(WORK / 'repair_inputs')
INPUTS = WORK / 'repair_inputs'
with zipfile.ZipFile(INPUTS/'alignment'/'alignment_checkpoint.zip') as z: z.extractall(INPUTS/'alignment'/'checkpoint')
with zipfile.ZipFile(INPUTS/'alignment'/'render_reports.zip') as z: z.extractall(INPUTS/'alignment'/'render_reports')
source_manifest = json.loads((INPUTS/'source'/'source_manifest.json').read_text(encoding='utf-8'))
render_manifest = json.loads((INPUTS/'alignment'/'render_reports'/'render_manifest.json').read_text(encoding='utf-8'))
print('Expected video duration:', render_manifest['video_duration_seconds'])
print('Expected base-video SHA256:', render_manifest['video_sha256'])
if abs(float(source_manifest['source_duration_seconds'])-float(render_manifest['video_duration_seconds'])) > 1.0:
    raise RuntimeError('Source manifest duration does not match render report.')
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()

expected_voices = source_manifest['voice_track_sha256_manifest']
for name, expected in expected_voices.items():
    voice_path = INPUTS/'voice_masters'/name
    if not voice_path.exists() or sha256_file(voice_path) != expected:
        raise RuntimeError(f'Voice master missing or hash mismatch: {name}')
print('All 10 voice master hashes verified.')



In [ ]:
# Download the existing dubbed base video (it already contains the correct picture + Arabic/music mix).
# MuseTalk clips will replace only the video stream; the original mixed audio is preserved bit-for-bit.
subprocess.run([sys.executable,'-m','pip','install','-q','gdown'],check=True)
BASE_VIDEO = WORK / 'base_dubbed.mkv'
if not BASE_VIDEO.exists() or BASE_VIDEO.stat().st_size != 971090724:
    import gdown
    gdown.download(id=BASE_VIDEO_ID, output=str(BASE_VIDEO), quiet=False)
if not BASE_VIDEO.exists() or BASE_VIDEO.stat().st_size < 900_000_000:
    raise RuntimeError('Base dubbed video download incomplete; do not proceed.')

actual_sha = sha256_file(BASE_VIDEO)
if actual_sha != render_manifest['video_sha256']:
    raise RuntimeError(f'Base video hash mismatch: {actual_sha}')
probe = json.loads(subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration','-of','json',str(BASE_VIDEO)]))
actual_duration=float(probe['format']['duration'])
if abs(actual_duration-float(render_manifest['video_duration_seconds'])) > 1.0:
    raise RuntimeError('Duration mismatch; source/alignment pair is not safe to use.')
print('Base video verified:', actual_duration, 'seconds; SHA256 matches.')


In [ ]:
# Clone the project branch and generate the conservative candidate plan.
repo_dir = WORK / 'AI-Creative-Studio'
if not repo_dir.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,f'https://github.com/{REPO}.git',str(repo_dir)],check=True)
plan_path = WORK/'lipsync_candidates.json'
subprocess.run([sys.executable,str(repo_dir/'scripts'/'prepare_musetalk_candidates.py'),
 '--render-manifest',str(INPUTS/'alignment'/'render_reports'/'render_manifest.json'),
 '--output',str(plan_path),'--min-confidence',str(MIN_CONFIDENCE),
 '--min-coverage',str(MIN_WORD_COVERAGE),'--min-audio-fit','0.75','--max-audio-fit','1.25','--max-clip-seconds',str(MAX_CLIP_SECONDS)],check=True)
plan=json.loads(plan_path.read_text(encoding='utf-8'))
print('Plan summary:', json.dumps(plan['summary'],ensure_ascii=False,indent=2))
print('Pilot clip:', plan['pilot_clip_id'])


## Install MuseTalk v1.5 (one-time model cache)

The setup uses Python 3.10 and CUDA 11.8-compatible PyTorch. Model weights are cached under `MyDrive/MuseTalkArabicCache/models`; future Colab sessions reuse them. The minimal downloader intentionally skips MuseTalk v1.0 and optional SyncNet weights.


In [ ]:
# Install a pinned Python 3.10 environment. This is setup only; no inference is run in this cell.
import os, shutil, subprocess
os.chdir('/content')
if not Path('/content/bin/micromamba').exists():
    subprocess.run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba",shell=True,cwd='/content',check=True)
if not Path('/content/musetalk-env/conda-meta').exists():
    subprocess.run(['/content/bin/micromamba','create','-y','-p','/content/musetalk-env','python=3.10','pip'],check=True)
if not Path('/content/MuseTalk').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/TMElyralab/MuseTalk.git','/content/MuseTalk'],check=True)
models_link=Path('/content/MuseTalk/models')
if models_link.is_symlink(): models_link.unlink()
elif models_link.exists(): shutil.rmtree(models_link)
models_link.symlink_to(CACHE/'models',target_is_directory=True)
commands=[
 ['/content/bin/micromamba','run','-p','/content/musetalk-env','pip','install','--upgrade','pip'],
 ['/content/bin/micromamba','run','-p','/content/musetalk-env','pip','install','torch==2.0.1','torchvision==0.15.2','--index-url','https://download.pytorch.org/whl/cu118'],
 ['/content/bin/micromamba','run','-p','/content/musetalk-env','pip','install','-r','/content/MuseTalk/requirements.txt'],
 ['/content/bin/micromamba','run','-p','/content/musetalk-env','pip','install','huggingface_hub[cli]==0.30.2','openmim','gdown'],
 ['/content/bin/micromamba','run','-p','/content/musetalk-env','mim','install','mmengine'],
 ['/content/bin/micromamba','run','-p','/content/musetalk-env','mim','install','mmcv==2.0.1'],
 ['/content/bin/micromamba','run','-p','/content/musetalk-env','mim','install','mmdet==3.1.0'],
 ['/content/bin/micromamba','run','-p','/content/musetalk-env','mim','install','mmpose==1.1.0'],
]
for command in commands: subprocess.run(command,check=True)


In [ ]:
# Download only MuseTalk v1.5 and its inference dependencies into persistent Drive cache.
required_weights = [CACHE/'models'/'musetalkV15'/'unet.pth', CACHE/'models'/'musetalkV15'/'musetalk.json', CACHE/'models'/'sd-vae'/'diffusion_pytorch_model.bin', CACHE/'models'/'whisper'/'pytorch_model.bin', CACHE/'models'/'dwpose'/'dw-ll_ucoco_384.pth', CACHE/'models'/'face-parse-bisent'/'79999_iter.pth']
models_ready = all(p.exists() for p in required_weights)
if not models_ready:
    env=os.environ.copy(); env['HF_ENDPOINT']='https://huggingface.co'
    env['MUSETALK_ROOT']='/content/MuseTalk'
    subprocess.run(['bash',str(repo_dir/'scripts'/'download_musetalk_v15_minimal.sh')],cwd='/content/MuseTalk',env=env,check=True)
else:
    print('Cached MuseTalk v1.5 weights found; skipping downloads.')


## Prepare and run the pilot

The first run processes one 4–7 second high-confidence utterance. It writes the result and coordinate cache directly to Drive so a disconnect does not discard completed work. Do not enable the full safe batch until the pilot is visually acceptable.


In [ ]:
import yaml, glob
VOICE_DIR = INPUTS/'voice_masters'
RESULT_DIR = CACHE/'musetalk_results'
RESULT_DIR.mkdir(parents=True,exist_ok=True)
(RESULT_DIR/'v15').mkdir(parents=True,exist_ok=True)
INPUT_DIR = WORK/'clip_inputs'; INPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT = CACHE/'completed_musetalk_clips.json'
completed = json.loads(CHECKPOINT.read_text()) if CHECKPOINT.exists() else {}

def ffmpeg_run(args):
    subprocess.run(['ffmpeg','-y','-v','error']+list(map(str,args)),check=True)

def prepare_clip(c):
    v=INPUT_DIR/f"{c['clip_id']}_video.mp4"
    a=INPUT_DIR/f"{c['clip_id']}_audio.wav"
    dur=float(c['target_duration']); vdur=dur
    if not v.exists():
        ffmpeg_run(['-ss',c['source_start'],'-i',BASE_VIDEO,'-t',vdur,'-an','-vf','fps=25','-c:v','libx264','-preset','ultrafast','-crf','22','-pix_fmt','yuv420p',v])
    voice=VOICE_DIR/c['voice_master']
    if not voice.exists(): raise FileNotFoundError(voice)
    if not a.exists():
        raw=float(c['voice_end'])-float(c['voice_start'])
        if c.get('silent_audio_chunk') or raw <= 0:
            ffmpeg_run(['-f','lavfi','-i','anullsrc=r=16000:cl=mono','-t',dur,'-c:a','pcm_s16le',a])
        else:
            af=f"atempo={c['atempo']},apad,atrim=duration={dur}"
            ffmpeg_run(['-ss',c['voice_start'],'-i',voice,'-t',raw,'-af',af,'-ac','1','-ar','16000','-c:a','pcm_s16le',a])
    return v,a

def run_musetalk_batch(pilot_only=True):
    clips=plan['clips']
    if pilot_only:
        pilot=next(c for c in clips if c['clip_id']==plan['pilot_clip_id'])
        pilot_out=RESULT_DIR/'v15'/f"{pilot['clip_id']}_lipsynced.mp4"
        pending=[] if Path(str(pilot_out)+'.done').exists() else [pilot]
    else:
        pending=[c for c in clips if not (RESULT_DIR/'v15'/f"{c['clip_id']}_lipsynced.mp4.done").exists()]
        pending=pending[:MAX_CLIPS_PER_SESSION]
    if not pending:
        print('No pending clips in this batch.'); return
    tasks={}
    for i,c in enumerate(pending):
        v,a=prepare_clip(c)
        tasks[f'task_{i}']={'video_path':str(v),'audio_path':str(a),'result_name':f"{c['clip_id']}_lipsynced.mp4"}
    cfg=WORK/'musetalk_batch.yaml'; cfg.write_text(yaml.safe_dump(tasks,allow_unicode=True),encoding='utf-8')
    cmd=['/content/bin/micromamba','run','-p','/content/musetalk-env','python','-m','scripts.inference',
         '--inference_config',str(cfg),'--result_dir',str(RESULT_DIR),'--unet_model_path','models/musetalkV15/unet.pth',
         '--unet_config','models/musetalkV15/musetalk.json','--version','v15','--gpu_id','0','--batch_size','2',
         '--ffmpeg_path','/usr/bin/ffmpeg','--use_float16','--saved_coord','--use_saved_coord','--max_runtime_seconds',str(MAX_SESSION_GPU_MINUTES*60)]
    start=time.time()
    # Patch the upstream loop to stop at the session budget and mark each completed clip.
    patcher=repo_dir/'scripts'/'patch_musetalk_resumable.py'
    subprocess.run([sys.executable,str(patcher),'/content/MuseTalk/scripts/inference.py'],check=True)
    subprocess.run(cmd,cwd='/content/MuseTalk',check=True,timeout=(MAX_SESSION_GPU_MINUTES+5)*60)
    elapsed=time.time()-start
    for c in pending:
        out=RESULT_DIR/'v15'/f"{c['clip_id']}_lipsynced.mp4"
        if out.exists() and out.stat().st_size>10000 and Path(str(out)+'.done').exists():
            completed[c['clip_id']]={'file':str(out),'seconds':c['target_duration'],'sha256':sha256_file(out),'completed_at_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
    CHECKPOINT.write_text(json.dumps(completed,ensure_ascii=False,indent=2),encoding='utf-8')
    # Reconcile outputs written before a runtime disconnect.
    for c in clips:
        out=RESULT_DIR/'v15'/f"{c['clip_id']}_lipsynced.mp4"
        if out.exists() and out.stat().st_size>10000 and Path(str(out)+'.done').exists():
            completed[c['clip_id']]={'file':str(out),'seconds':c['target_duration'],'sha256':sha256_file(out)}
    CHECKPOINT.write_text(json.dumps(completed,ensure_ascii=False,indent=2),encoding='utf-8')
    print('Batch seconds:',round(elapsed,1),'completed:',len(completed),'of',len(clips))
    if pilot_only: print('PILOT ONLY: inspect the sample before setting PILOT_ONLY=False.')

run_musetalk_batch(pilot_only=PILOT_ONLY)


In [ ]:
# Display the pilot output for visual review.
from IPython.display import Video, display
pilot_path=RESULT_DIR/'v15'/f"{plan['pilot_clip_id']}_lipsynced.mp4"
if pilot_path.exists(): display(Video(str(pilot_path),embed=True))
else: print('Pilot output missing; inspect MuseTalk logs before retrying.')


## Resume batches after approving the pilot

After reviewing the pilot's face selection, mouth motion and audio synchronization, change `PILOT_ONLY` to `False` and rerun the batch cell. Each run processes at most six remaining clips. Completed MP4s and checkpoints are stored in Drive. Re-run the cell after a Colab disconnect to resume; already completed clips are skipped.


In [ ]:
# Keep this False until the pilot is visually approved.
APPROVE_FULL_BATCH = False
if APPROVE_FULL_BATCH:
    run_musetalk_batch(pilot_only=False)
else:
    print('Stopped after pilot by design. Set APPROVE_FULL_BATCH=True only after visual approval.')


## Assemble and upload when all safe candidates are complete

This overlays only completed MuseTalk video clips onto the verified dubbed base. It maps the original Arabic/music audio from the base file unchanged; it does not mux the individual pilot audio tracks. If some clips are missing, the output is explicitly marked partial.


In [ ]:
# Assembly is refused unless all 113 conservative clips exist, unless ALLOW_PARTIAL is explicitly set.
ALLOW_PARTIAL = False
completed_paths=[RESULT_DIR/'v15'/f"{c['clip_id']}_lipsynced.mp4" for c in plan['clips']]
missing=[p.name for p in completed_paths if not p.exists() or not Path(str(p)+'.done').exists()]
if missing and not ALLOW_PARTIAL:
    print(f'Not ready: {len(missing)} of {len(plan["clips"])} clips are missing. Resume batches first.')
else:
    output=WORK/('الحلقة_مدبلجة_مزامنة_شفاه_جزئية.mkv' if missing else 'الحلقة_مدبلجة_مزامنة_شفاه_مقاطع_موثوقة.mkv')
    subprocess.run([sys.executable,str(repo_dir/'scripts'/'assemble_musetalk_overlays.py'),
        '--base-video',str(BASE_VIDEO),'--plan',str(plan_path),'--clips-dir',str(RESULT_DIR/'v15'),
        '--output',str(output)]+(['--allow-partial'] if ALLOW_PARTIAL else []),check=True)
    print('Output:',output,'bytes:',output.stat().st_size)


In [ ]:
# Upload the resulting MKV to the same Drive folder using resumable upload.
# This cell requires the user to approve Google Drive OAuth inside Colab.
if 'output' in globals() and output.exists():
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload
    service=build('drive','v3')
    body={'name':output.name,'parents':[OUTPUT_FOLDER_ID],'mimeType':'video/x-matroska'}
    media=MediaFileUpload(str(output),mimetype='video/x-matroska',resumable=True,chunksize=8*1024*1024)
    req=service.files().create(body=body,media_body=media,fields='id,name,size,webViewLink')
    response=None
    while response is None:
        status,response=req.next_chunk()
        if status: print('Drive upload:',round(status.progress()*100,1),'%')
    print('Uploaded:',response)
else:
    print('No assembled output to upload yet.')
